In [ ]:
tenant_id = "acme"
batch_date = "2026-06-11"

In [ ]:
%pip install openai 'azure-search-documents>=11.4.0' azure-keyvault-secrets azure-identity

In [ ]:
import sys
sys.path.insert(0, "/home/trusted-service-user/work/meridian")

In [ ]:
import time
from datetime import datetime, timezone

from azure.core.credentials import AzureKeyCredential
from azure.identity import ManagedIdentityCredential
from azure.keyvault.secrets import SecretClient
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    HnswAlgorithmConfiguration,
    HnswParameters,
    SearchField,
    SearchFieldDataType,
    SearchIndex,
    SearchableField,
    SemanticConfiguration,
    SemanticField,
    SemanticPrioritizedFields,
    SemanticSearch,
    SimpleField,
    VectorSearch,
    VectorSearchAlgorithmMetric,
    VectorSearchProfile,
)
from openai import AzureOpenAI
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, FloatType

In [ ]:
KV_URL = "https://meridian-kv-rk1.vault.azure.net/"
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIMS = 1536
INDEX_NAME = "meridian-docs"
BATCH_SIZE = 16
# text-embedding-3-small enforces per-minute request limits - sleep prevents 429s at dev volumes
SLEEP_BETWEEN_BATCHES = 1.0

mi_credential = ManagedIdentityCredential()
kv = SecretClient(vault_url=KV_URL, credential=mi_credential)

openai_endpoint = kv.get_secret("openai-endpoint").value
openai_key      = kv.get_secret("openai-key").value
search_endpoint = kv.get_secret("ai-search-endpoint").value
search_key      = kv.get_secret("ai-search-key").value

oai = AzureOpenAI(
    azure_endpoint=openai_endpoint,
    api_key=openai_key,
    api_version="2024-02-01",
)
search_cred = AzureKeyCredential(search_key)

In [ ]:
silver_df = (
    spark.table("silver_doc_chunks")
    .filter(
        (F.col("tenant_id") == tenant_id)
        & (F.col("batch_date") == batch_date)
        & (F.col("dq_passed") == True)
    )
)
silver_rows = silver_df.collect()
print(f"chunks to embed: {len(silver_rows)}")

In [ ]:
def _embed_batch(texts: list[str]) -> list[list[float]]:
    response = oai.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts,
        dimensions=EMBEDDING_DIMS,
    )
    return [item.embedding for item in response.data]

contents = [r["content"] for r in silver_rows]
embeddings: list[list[float]] = []

for i in range(0, len(contents), BATCH_SIZE):
    batch = contents[i : i + BATCH_SIZE]
    embeddings.extend(_embed_batch(batch))
    if i + BATCH_SIZE < len(contents):
        time.sleep(SLEEP_BETWEEN_BATCHES)

print(f"embeddings generated: {len(embeddings)}")

In [ ]:
gold_enriched_at = datetime.now(timezone.utc).isoformat()

enriched_records = []
for row, embedding in zip(silver_rows, embeddings):
    record = row.asDict()
    record["embedding"] = embedding
    record["gold_enriched_at"] = gold_enriched_at
    record["embedding_model"] = EMBEDDING_MODEL
    enriched_records.append(record)

In [ ]:
gold_df = spark.createDataFrame(enriched_records)
# overwriteSchema needed - gold adds embedding + gold_enriched_at columns absent from silver
gold_df = gold_df.withColumn("embedding", F.col("embedding").cast(ArrayType(FloatType())))

gold_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_doc_chunks_enriched")

print(f"gold_doc_chunks_enriched written: {gold_df.count()} rows")

In [ ]:
index_client = SearchIndexClient(endpoint=search_endpoint, credential=search_cred)

fields = [
    SimpleField(name="chunk_id",       type=SearchFieldDataType.String, key=True, filterable=True),
    SimpleField(name="doc_id",         type=SearchFieldDataType.String, filterable=True),
    SimpleField(name="tenant_id",      type=SearchFieldDataType.String, filterable=True, facetable=True),
    SimpleField(name="doc_type",       type=SearchFieldDataType.String, filterable=True, facetable=True),
    SimpleField(name="batch_date",     type=SearchFieldDataType.String, filterable=True),
    SimpleField(name="section_idx",    type=SearchFieldDataType.Int32,  filterable=True),
    SimpleField(name="content_length", type=SearchFieldDataType.Int32,  filterable=True),
    SearchableField(name="content",    type=SearchFieldDataType.String),
    SearchField(
        name="content_vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        retrievable=False,
        # vectors never need to round-trip back to the API layer - cuts response payload
        vector_search_dimensions=1536,
        vector_search_profile_name="meridian-hnsw-profile",
    ),
    SimpleField(name="gold_enriched_at", type=SearchFieldDataType.DateTimeOffset, filterable=True),
]

vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(
            name="meridian-hnsw",
            parameters=HnswParameters(
                m=4,
                ef_construction=400,
                ef_search=500,
                metric=VectorSearchAlgorithmMetric.COSINE,
            ),
        )
    ],
    profiles=[VectorSearchProfile(name="meridian-hnsw-profile", algorithm_configuration_name="meridian-hnsw")],
)

semantic_search = SemanticSearch(
    configurations=[
        SemanticConfiguration(
            name="meridian-semantic",
            prioritized_fields=SemanticPrioritizedFields(
                content_fields=[SemanticField(field_name="content")],
                keywords_fields=[SemanticField(field_name="doc_type"), SemanticField(field_name="tenant_id")],
            ),
        )
    ]
)

index = SearchIndex(
    name=INDEX_NAME,
    fields=fields,
    vector_search=vector_search,
    semantic_search=semantic_search,
)

try:
    index_client.get_index(INDEX_NAME)
    print(f"index {INDEX_NAME} already exists")
except Exception:
    index_client.create_index(index)
    print(f"index {INDEX_NAME} created")

In [ ]:
search_client = SearchClient(
    endpoint=search_endpoint,
    index_name=INDEX_NAME,
    credential=search_cred,
)

search_docs = []
for row, embedding in zip(silver_rows, embeddings):
    search_docs.append({
        "chunk_id":         row["chunk_id"],
        "doc_id":           row["doc_id"],
        "tenant_id":        row["tenant_id"],
        "doc_type":         row["doc_type"],
        "batch_date":       str(row["batch_date"]),
        "section_idx":      int(row["section_idx"]),
        "content_length":   int(row["content_length"]),
        "content":          row["content"],
        "content_vector":   embedding,
        "gold_enriched_at": gold_enriched_at,
    })

# AI Search caps batch size at 1000 - using 100 to stay well under limit
UPLOAD_BATCH = 100
succeeded = 0
for i in range(0, len(search_docs), UPLOAD_BATCH):
    results = search_client.merge_or_upload_documents(search_docs[i : i + UPLOAD_BATCH])
    succeeded += sum(1 for r in results if r.succeeded)

print(f"upserted to {INDEX_NAME}: {succeeded}/{len(search_docs)} documents")

In [ ]:
print(f"tenant:       {tenant_id}")
print(f"batch_date:   {batch_date}")
print(f"embedded:     {len(embeddings)} chunks")
print(f"gold table:   gold_doc_chunks_enriched")
print(f"search index: {INDEX_NAME}")